In [141]:
import pandas as pd
import re
import numpy as np
df = pd.read_csv("naukari.csv")


## Salary Processing Pipeline

1. Read `naukari.csv` into `df`.
2. Define converters:
  - `convert_dollar_to_lacs(salary)`:
    - strip "$", commas
    - extract numeric values
    - multiply by 80 (INR conversion)
    - format as `low-high` or single value
  - `convert_cr_to_lacs(salary)`:
    - strip "₹"
    - extract numeric values
    - multiply by 1e7
    - format as range or single value
  - `convert_salary_value(x)`:
    - if contains `"Cr"`, call `convert_cr_to_lacs`
    - elif contains `"$"`, call `convert_dollar_to_lacs`
    - else clean up
     - strip commas, quotes, `₹`, `P.A.`, `Lacs`
     - parse hyphen range
     - if numeric and <500, multiply by 100000 (to lacs→annual INR)
     - return `"low-high"` (or same for fixed)
    - non-string: `"Not Disclosed"`

3. Apply to `df["salary"]`:
  - `df["salary"] = df["salary"].apply(convert_salary_value)`

4. Export cleaned values:
  - `df["salary"].to_csv("cleaned_salaries.csv", index=False)`

In [142]:
#convert salary values to lacs
def convert_dollar_to_lacs(salary):
  if pd.isna(salary) or not isinstance(salary, str):
    return salary
  salary = salary.replace("$", "").strip()
  salary = salary.replace(",", "")
  is_above = "and above" in salary
  numbers = re.findall(r"\d+\.?\d*", salary)
  numbers_lacs = [float(num) * 80 for num in numbers]
  if len(numbers_lacs) == 2:
    result = f"{numbers_lacs[0]}-{numbers_lacs[1]}"
  elif len(numbers_lacs) == 1:
    result = f"{numbers_lacs[0]}"
  else:
    result = salary
  if result== "":
    print(salary)
  return result

#convert salary values in crores to lacs
def convert_cr_to_lacs(salary):
  if pd.isna(salary) or not isinstance(salary, str):
    return salary
  salary = salary.replace("₹", "").strip()
  is_above = "and above" in salary
  numbers = re.findall(r"\d+\.?\d*", salary)
  numbers_lacs = [float(num) * 10000000 for num in numbers]
  if len(numbers_lacs) == 2:
    result = f"{numbers_lacs[0]}-{numbers_lacs[1]}"
  elif len(numbers_lacs) == 1:
    result = f"{numbers_lacs[0]}"
  else:
    result = salary
  return result

def convert_salary_value(x):
    if isinstance(x, str) and "Not Disclosed" in x:
        return "Not Disclosed-Not Disclosed"
    if isinstance(x, str) and "Cr" in x:
        return convert_cr_to_lacs(x)
    elif isinstance(x, str) and "$" in x:
        return convert_dollar_to_lacs(x)
    elif isinstance(x, str):
        cleanrange = x.replace(',', '').strip().replace('"', '').strip().replace("₹", "").strip().replace("P.A.", "").strip().replace("Lacs", "").strip().split('(')[0].strip()
        if cleanrange.find('-') != -1:
            parts = cleanrange.split('-')
            if len(parts) == 2:
                try:
                  low = float(parts[0])
                  high = float(parts[1])
                  if low < 500:
                    low *= 100000
                  if high < 500:
                    high *= 100000
                  cleanrange = f"{low}-{high}"
                except ValueError:
                  cleanrange = f"{parts[0]}-{parts[1]}"
            elif len(parts) == 1:
              cleanrange = f"{parts[0]}"
        else:
            try:
              value = float(cleanrange)
              if value < 500:
                value *= 100000
              cleanrange = f"{value}-{value}"
            except ValueError:
              cleanrange = f"{cleanrange}"
        return cleanrange
    else:
        return "Not Disclosed-Not Disclosed"

df["salary"] = df["salary"].apply(convert_salary_value).astype(str)
df[["min_salary", "max_salary"]] = df["salary"].str.split("-", expand=True)

df.head(10000)

# 

,Unnamed: 0,job_title,company,rating,exp_req,salary,location,post_date,applicants,role,industry_type,department,emp_type,role_category,UG,key_skills,min_salary,max_salary
0,0,Probationary Officers' Programme,Icici Bank,4.0,0 - 5 years,425000.0-450000.0,"['Kolkata,', 'Hyderabad/Secunderabad,', 'Pune,...",Posted: Few Hours Ago,Job Applicants: 88751,Retail Banking Sales,Banking,Sales & Business Development,"Full Time, Permanent",Retail & B2C Sales,Any Graduate,"['Graduate', 'Customer Service', 'Sales', 'Bfs...",425000.0,450000.0
1,1,Editorial Reviewer: Medicine and Life Sciences...,CACTUS Communications,3.5,0 - 4 years,Not Disclosed-Not Disclosed,"['Kolkata,', 'Pune,', 'Chennai,', 'Bangalore/B...",Posted: 1 day ago,Job Applicants: 88751,Editor / Content Analyst,Pharmaceutical & Life Sciences,"Content, Editorial & Journalism","Full Time, Temporary/Contractual",Editing (Print / Online / Electronic),"BDS in Any Specialization, B.Pharma in Any Spe...","['editing', 'Medicine', 'Biotechnology', 'Cont...",Not Disclosed,Not Disclosed
2,2,Job Opening | Chat Process | Day Shift | MNC,Trigent Software,3.7,0 years,Not Disclosed-Not Disclosed,"['Kolkata,', 'Hyderabad/Secunderabad,', 'Pune,...",Posted: 1 day ago,Job Applicants: 687,Email Support,BPO / Call Centre,"Customer Success, Service & Operations","Full Time, Permanent",Non Voice,Any Graduate,"['Non Voice', 'Chat Process']",Not Disclosed,Not Disclosed
3,3,Customer Support Executive,Careernet Technologies,NaN,0 - 5 years,250000.0-475000.0,"['Bangalore/Bengaluru,', 'Mumbai', '(All', 'Ar...",Posted: 1 day ago,Job Applicants: Less than 10,Customer Retention - Voice / Blended,Banking,"Customer Success, Service & Operations","Full Time, Permanent",Voice / Blended,Graduation Not Required,"['international voice process', 'International...",250000.0,475000.0
4,4,Top most International Bpo Hiring with the bes...,Careernet Technologies,4.2,0 - 3 years,250000.0-400000.0,"['Bangalore/Bengaluru,', 'Mumbai', '(All', 'Ar...",Posted: 2 days ago,Job Applicants: 17,Customer Service,BPO / Call Centre,"Customer Success, Service & Operations","Full Time, Permanent","Customer Success, Service & Operations - Other",Any Graduate,"['international voice process', 'US Process', ...",250000.0,400000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,10936,ITOM ServiceNow Developer,Saven Nova Technologies,3.3,8 - 13 years,2000000.0-3500000.0,"['Noida,', 'Hyderabad/Secunderabad,', 'Pune,',...",Posted: 8 days ago,Job Applicants: 92,IT & Information Security - Other,IT Services & Consulting,IT & Information Security,"Full Time, Temporary/Contractual",IT & Information Security - Other,Any Graduate,"['ITOM', 'ServiceNow', 'IT Operations Manageme...",2000000.0,3500000.0
9996,10937,Business Development Executive,Quick Clean,3.3,0 - 2 years,200000.0-600000.0,"['Kolkata,', 'Chandigarh,', 'Hyderabad/Secunde...",Posted: 8 days ago,Job Applicants: 55,Business Development Executive (BDE),Electronics Manufacturing,Sales & Business Development,"Full Time, Permanent",BD / Pre Sales,B.B.A/ B.M.S in Any Specialization,"['Sales', 'Business Development']",200000.0,600000.0
9997,10938,Java Microservices,Experis,3.8,4 - 9 years,1900000.0-2250000.0,"['Hyderabad/Secunderabad,', 'Pune,', 'Gurgaon/...",Posted: 8 days ago,Job Applicants: 235,IT & Information Security - Other,IT Services & Consulting,IT & Information Security,"Full Time, Permanent",IT & Information Security - Other,Any Graduate,"['Microservices', 'OAUTH', 'Java Spring Boot',...",1900000.0,2250000.0
9998,10939,Cloud Technical Support Engineer,Puresoftware,3.6,3 - 8 years,Not Disclosed-Not Disclosed,"['Hyderabad/Secunderabad,', 'Pune,', 'Bangalor...",Posted: 8 days ago,Job Applicants: 903,IT Support - Other,IT Services & Consulting,IT & Information Security,"Full Time, Permanent",IT Support,"B.Tech/B.E. in Any Specialization, BCA in Any ...","['oracle support', 'technical support', 'it su...",Not Disclosed,Not Disclosed
